# Phase 10-11: Generation & Evaluation

Inference dan evaluasi memakai sitasi natural dari schema chunk baru, tanpa ID `[R#]` di jawaban.


## Mapping Fase ke Implementasi Saat Ini

Pipeline produksi di `pipeline_legal_rag_indonesia.md` mendefinisikan Phase 1-11. Implementasi notebook ini memakai varian praktis ChromaDB:

- Phase 1-3: dikerjakan oleh `praproses_ringan_pasal.py`.
- Phase 4: dikerjakan oleh `finalize_chunks_for_chroma.py`, menghasilkan `../data/processed_chunks_ringan_pasal_chroma_ready.json`.
- Phase 5: embedding memakai `embedding_text` dengan `intfloat/multilingual-e5-base`.
- Phase 6: vector store memakai ChromaDB lokal sesuai proposal final.
- Phase 7: hybrid retrieval memakai dense Chroma + BM25 lokal.
- Phase 8: reranker dibuat opsional. Default mati supaya notebook ringan.
- Phase 9: context assembler memakai `display_text` + `citation_text`; sibling expansion bisa ditambah setelah chunk ayat/sibling tersedia stabil.
- Phase 10: generation wajib menyebut sumber hukum secara natural, misalnya `Peraturan Pemerintah No. 35 Tahun 2021, Pasal 56`.
- Phase 11: evaluasi inference + hook RAGAS.


In [ ]:
import hashlib
import json
import os
import pickle
import re
from pathlib import Path
from typing import Dict, List, Any, Set

import chromadb
import numpy as np
import torch
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

DATA_PATH = Path("../data/processed_chunks_ringan_pasal_chroma_ready.json")
CHROMA_DB_DIR = Path("../data/chroma_db")
COLLECTION_NAME = "hukum_ketenagakerjaan"
BM25_PATH = Path("../data/bm25_index.pkl")
EMBEDDING_MODEL_NAME = "intfloat/multilingual-e5-base"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {DEVICE}")
print(f"Data: {DATA_PATH}")
print(f"Chroma: {CHROMA_DB_DIR} / {COLLECTION_NAME}")

In [ ]:
def load_chunks(path: Path = DATA_PATH) -> List[Dict[str, Any]]:
    if not path.exists():
        raise FileNotFoundError(f"{path} tidak ditemukan. Jalankan praproses_ringan_pasal.py lalu finalize_chunks_for_chroma.py dulu.")
    chunks = json.loads(path.read_text(encoding="utf-8"))
    required = {"id", "text", "display_text", "embedding_text", "citation_text", "metadata"}
    missing = [i for i, c in enumerate(chunks[:20]) if not required.issubset(c)]
    if missing:
        raise ValueError(f"Chunk belum pakai schema baru. Cek index sample: {missing}")
    return ensure_unique_chunk_ids(chunks)


def ensure_unique_chunk_ids(chunks: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen = {}
    fixed = 0
    for idx, chunk in enumerate(chunks):
        base_id = str(chunk.get("id") or f"chunk-{idx}")
        count = seen.get(base_id, 0)
        seen[base_id] = count + 1
        if count:
            seed = "::".join([
                base_id,
                str(idx),
                chunk.get("metadata", {}).get("source_file", ""),
                chunk.get("metadata", {}).get("pasal_id", ""),
                chunk.get("text", "")[:200],
            ])
            chunk["original_id"] = base_id
            chunk["id"] = hashlib.sha1(seed.encode("utf-8")).hexdigest()
            fixed += 1
    if fixed:
        print(f"Fixed duplicated chunk ids in memory: {fixed}")
    return chunks


def normalize_metadata(chunk: Dict[str, Any]) -> Dict[str, Any]:
    meta = dict(chunk.get("metadata", {}))
    meta["chunk_id"] = chunk.get("id", "")
    meta["citation_text"] = chunk.get("citation_text", "")
    meta["source_file"] = meta.get("source_file", "")
    meta["pasal_id"] = meta.get("pasal_id", "")
    meta["bab"] = meta.get("bab", "")
    meta["bab_title"] = meta.get("bab_title", "")
    meta["regulation_type"] = meta.get("regulation_type", "")
    meta["nomor"] = meta.get("nomor", "")
    meta["tentang"] = meta.get("tentang", "")
    meta["year"] = int(meta.get("year") or 0)
    meta["publication_year"] = int(meta.get("publication_year") or meta.get("year") or 0)
    safe = {}
    for key, value in meta.items():
        if value is None:
            safe[key] = ""
        elif isinstance(value, (str, int, float, bool)):
            safe[key] = value
        else:
            safe[key] = json.dumps(value, ensure_ascii=False) if isinstance(value, (dict, list)) else str(value)
    return safe


def tokenize_for_bm25(text: str) -> List[str]:
    return re.findall(r"[a-zA-Z0-9]+", text.lower())


def compact_citation(meta: Dict[str, Any]) -> str:
    reg_type = str(meta.get("regulation_type") or "Aturan").strip()
    nomor = str(meta.get("nomor") or "").strip()
    year = str(meta.get("publication_year") or meta.get("year") or "").strip()
    pasal = str(meta.get("pasal_id") or "").strip()

    parts = [reg_type]
    if nomor and nomor.lower() != "unknown":
        parts.append(f"No. {nomor}")
    if year and year != "0":
        parts.append(f"Tahun {year}")
    citation = " ".join(parts).strip()
    if pasal:
        citation = f"{citation}, {pasal}"
    return citation


def build_reference(meta: Dict[str, Any], idx: int | None = None) -> str:
    # idx dipertahankan untuk kompatibilitas cell lama, tapi output tetap sitasi natural.
    return compact_citation(meta)

chunks = load_chunks()
print(f"Loaded {len(chunks)} chunks from {DATA_PATH}")
print("Sample citation:", chunks[0]["citation_text"])

In [ ]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=DEVICE)
client = chromadb.PersistentClient(path=str(CHROMA_DB_DIR))
collection = client.get_collection(COLLECTION_NAME)

bm25 = None
bm25_ids = []
if BM25_PATH.exists():
    with BM25_PATH.open("rb") as f:
        payload = pickle.load(f)
    bm25 = payload["bm25"]
    bm25_ids = payload["ids"]
    print(f"BM25 loaded: {len(bm25_ids)} docs")
else:
    print("BM25 index tidak ditemukan. Retrieval tetap jalan dengan dense Chroma saja.")


def dense_search(query: str, fetch_k: int = 30) -> List[Dict[str, Any]]:
    q_emb = embedding_model.encode(
        ["query: " + query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )[0].tolist()
    res = collection.query(
        query_embeddings=[q_emb],
        n_results=fetch_k,
        include=["documents", "metadatas", "distances"],
    )
    hits = []
    for doc_id, doc, meta, dist in zip(res["ids"][0], res["documents"][0], res["metadatas"][0], res["distances"][0]):
        hits.append({"id": doc_id, "text": doc, "metadata": meta, "dense_distance": float(dist), "source": "dense"})
    return hits


def bm25_search(query: str, fetch_k: int = 30) -> List[Dict[str, Any]]:
    if bm25 is None:
        return []
    scores = bm25.get_scores(tokenize_for_bm25(query))
    order = np.argsort(scores)[::-1][:fetch_k]
    ids = [bm25_ids[i] for i in order if scores[i] > 0]
    if not ids:
        return []
    got = collection.get(ids=ids, include=["documents", "metadatas"])
    lookup = {doc_id: (doc, meta) for doc_id, doc, meta in zip(got["ids"], got["documents"], got["metadatas"])}
    hits = []
    for i in order:
        doc_id = bm25_ids[i]
        if scores[i] <= 0 or doc_id not in lookup:
            continue
        doc, meta = lookup[doc_id]
        hits.append({"id": doc_id, "text": doc, "metadata": meta, "bm25_score": float(scores[i]), "source": "bm25"})
    return hits


def rrf_fuse(result_sets: List[List[Dict[str, Any]]], weights: List[float] | None = None, rrf_k: int = 60) -> List[Dict[str, Any]]:
    weights = weights or [1.0] * len(result_sets)
    fused = {}
    for hits, weight in zip(result_sets, weights):
        for rank, hit in enumerate(hits, 1):
            item = fused.setdefault(hit["id"], {"score": 0.0, "hit": hit})
            item["score"] += weight / (rrf_k + rank)
    out = []
    for item in sorted(fused.values(), key=lambda x: x["score"], reverse=True):
        hit = item["hit"]
        hit["rrf_score"] = item["score"]
        out.append(hit)
    return out

# Kata-kata yang menandakan regulasi memang berkaitan ketenagakerjaan / hubungan kerja.
KETENAGAKERJAAN_POSITIVE_KEYWORDS: Set[str] = {
    "ketenagakerjaan", "tenaga kerja", "pekerja", "buruh", "hubungan kerja",
    "perjanjian kerja", "pemutusan hubungan kerja", "phk", "pesangon",
    "upah", "pengupahan", "pensiun", "jaminan sosial", "jaminan kerja",
    "alih daya", "outsourcing", "pkwt", "pkwtt", "serikat pekerja",
    "pengusaha", "perusahaan alih daya", "perlindungan pekerja",
    "waktu kerja", "cuti", "k3", "keselamatan kerja", "cipta kerja",
}
 
# Kata-kata yang PASTI bukan ketenagakerjaan — tolak langsung jika ada.
KETENAGAKERJAAN_NEGATIVE_KEYWORDS: Set[str] = {
    "perizinan berusaha", "oss", "risiko usaha", "izin usaha", "nib",
    "investasi", "badan usaha", "penyelenggaraan usaha", "sistem perizinan",
    "rba", "risk based approach", "sektor usaha", "kbli",
    "penyelenggaraan pemerintahan", "administrasi pemerintahan",
    "sop administrasi", "pelayanan publik",
}
 
# Whitelist nomor PP/UU ketenagakerjaan yang diketahui valid.
# Ini sebagai safety net kedua — dokumen ini PASTI boleh masuk walau keyword tidak cocok.
KETENAGAKERJAAN_KNOWN_REGS: Set[tuple] = {
    # (regulation_type_normalized, nomor)
    ("pp", "35"),   # PP No. 35 Tahun 2021 — Perjanjian Kerja, PHK, Kompensasi
    ("pp", "36"),   # PP No. 36 Tahun 2021 — Pengupahan
    ("pp", "34"),   # PP No. 34 Tahun 2021 — TKA
    ("pp", "45"),   # PP No. 45 Tahun 2015 — Jaminan Pensiun BPJS
    ("uu", "13"),   # UU No. 13 Tahun 2003 — Ketenagakerjaan
    ("uu", "6"),    # UU No. 6 Tahun 2023 — Cipta Kerja (klaster ketenagakerjaan)
    ("uu", "24"),   # UU No. 24 Tahun 2011 — BPJS
    ("uu", "1"),    # UU No. 1 Tahun 1970 — Keselamatan Kerja
    ("uu", "21"),   # UU No. 21 Tahun 2000 — Serikat Pekerja
    ("permen", "5"),  # Permenaker terkait PHK
    ("permen", "6"),
}
 
 
def _normalize_reg_type(value: str) -> str:
    """Normalisasi tipe regulasi ke string pendek."""
    value = str(value or "").lower()
    if "undang" in value or value == "uu":
        return "uu"
    if "pemerintah" in value or value == "pp":
        return "pp"
    if "presiden" in value or "perpres" in value:
        return "perpres"
    if "menteri" in value or "permen" in value:
        return "permen"
    return value
 
 
def is_ketenagakerjaan_doc(meta: Dict[str, Any]) -> bool:
    """
    Mengembalikan True jika dokumen termasuk domain ketenagakerjaan.
 
    Urutan pemeriksaan:
    1. Cek negative keywords di field 'tentang' → tolak langsung.
    2. Cek known_regs whitelist → lolos langsung.
    3. Cek positive keywords di field 'tentang' atau 'bab_title' → lolos jika ada.
    4. Default: tolak (fail-safe).
    """
    tentang = str(meta.get("tentang", "") or "").lower()
    bab_title = str(meta.get("bab_title", "") or "").lower()
    bagian_title = str(meta.get("bagian_title", "") or "").lower()
    haystack = f"{tentang} {bab_title} {bagian_title}"
 
    # 1. Tolak cepat berdasarkan negative keyword.
    if any(neg in haystack for neg in KETENAGAKERJAAN_NEGATIVE_KEYWORDS):
        return False
 
    # 2. Whitelist nomor regulasi yang diketahui.
    reg_type = _normalize_reg_type(meta.get("regulation_type", ""))
    nomor = str(meta.get("nomor", "") or "").strip()
    if (reg_type, nomor) in KETENAGAKERJAAN_KNOWN_REGS:
        return True
 
    # 3. Positive keyword match.
    if any(pos in haystack for pos in KETENAGAKERJAAN_POSITIVE_KEYWORDS):
        return True
 
    # 4. Jika 'tentang' kosong (data tidak lengkap), beri benefit of the doubt
    #    HANYA jika nomor regulasinya tidak diketahui sama sekali.
    if not tentang.strip():
        return True  # data tidak lengkap — jangan buang, biarkan filter lain yang putuskan
 
    return False

def lex_posterior_score(hit: Dict[str, Any]) -> float:
    meta = hit.get("metadata", {})
    score = float(hit.get("rrf_score", 0.0))
 
    try:
        year = int(meta.get("publication_year") or meta.get("year") or 0)
    except Exception:
        year = 0
    try:
        hierarchy = int(meta.get("regulation_hierarchy") or 99)
    except Exception:
        hierarchy = 99
 
    if str(meta.get("active_status", "")).lower() == "berlaku":
        score += 0.030
 
    # FIX: year bonus hanya untuk dokumen ketenagakerjaan yang valid.
    # Sebelumnya: score += min(max(year - 2000, 0), 40) * 0.001  ← buta domain
    # Sekarang: PP 28/2025 (OSS) tidak mendapat year bonus ini.
    if is_ketenagakerjaan_doc(meta):
        score += min(max(year - 2000, 0), 40) * 0.001
    else:
        # Penalti keras untuk dokumen lintas-domain yang lolos ke retrieval.
        score -= 0.100
 
    score += max(0, 6 - hierarchy) * 0.003
 
    if meta.get("quality_status") == "needs_review":
        score -= 0.010
 
    return score


def dedupe_legal_hits(hits: List[Dict[str, Any]], k: int = 8) -> List[Dict[str, Any]]:
    ranked = sorted(hits, key=lex_posterior_score, reverse=True)
    seen: set = set()
    out: List[Dict[str, Any]] = []
 
    for hit in ranked:
        meta = hit.get("metadata", {})
 
        # FIX: buang dokumen lintas-domain sebelum masuk ke konteks.
        if not is_ketenagakerjaan_doc(meta):
            continue
 
        key = (
            meta.get("source_file", ""),
            meta.get("pasal_id", ""),
            meta.get("chunk_kind", ""),
            meta.get("chunk_index", ""),
        )
        if key in seen:
            continue
        seen.add(key)
        hit["final_score"] = lex_posterior_score(hit)
        out.append(hit)
        if len(out) >= k:
            break
    return out


def retrieve_documents(query: str, k: int = 6, fetch_k: int = 30, use_bm25: bool = True) -> List[Dict[str, Any]]:
    dense_hits = dense_search(query, fetch_k=fetch_k)
    sparse_hits = bm25_search(query, fetch_k=fetch_k) if use_bm25 else []
    fused = rrf_fuse([dense_hits, sparse_hits], weights=[1.0, 0.7]) if sparse_hits else dense_hits
    return dedupe_legal_hits(fused, k=k)

def _extract_cited_reg_keys(validation: Dict[str, Any]) -> Set[tuple]:
    """Parsing validation['cited'] → set of (type_normalized, nomor, year)."""
    pattern = re.compile(
        r"(Undang-Undang|Peraturan Pemerintah|Peraturan Presiden|Peraturan Menteri(?:\s+Ketenagakerjaan)?|UU|PP|Perpres|Permen)"
        r"\s+No\.\s*([A-Za-z0-9./-]+)\s+Tahun\s+(\d{4})",
        re.IGNORECASE,
    )
    keys: Set[tuple] = set()
    for label in validation.get("cited", []):
        m = pattern.match(label)
        if m:
            keys.add((_normalize_reg_type(m.group(1)), m.group(2).strip(), m.group(3)))
    return keys

def cited_docs_only(docs: List[Dict[str, Any]], validation: Dict[str, Any]) -> List[Dict[str, Any]]:
    """Filter docs: hanya kembalikan doc yang regulasinya benar-benar dikutip LLM."""
    cited_keys = _extract_cited_reg_keys(validation)
    if not cited_keys:
        return []
    out = []
    seen_pasal: set = set()
    for doc in docs:
        meta = doc.get("metadata", {})
        key = (
            _normalize_reg_type(meta.get("regulation_type", "")),
            str(meta.get("nomor", "")).strip(),
            str(meta.get("publication_year") or meta.get("year") or ""),
        )
        if key not in cited_keys:
            continue
        pasal_key = (key, str(meta.get("pasal_id", "")).strip())
        if pasal_key in seen_pasal:
            continue
        seen_pasal.add(pasal_key)
        out.append(doc)
    return out

def print_references(docs: List[Dict[str, Any]], validation: Dict[str, Any] | None = None) -> None:
    """
    Cetak referensi.
    Jika validation diberikan, hanya cetak dokumen yang dikutip dalam teks jawaban.
    Jika validation=None (backward-compat), cetak semua seperti sebelumnya.
    """
    if validation is not None:
        docs = cited_docs_only(docs, validation)
    seen: set = set()
    for doc in docs:
        citation = compact_citation(doc["metadata"])
        if citation in seen:
            continue
        seen.add(citation)
        print(f"- {citation}")

# Smoke test
smoke_docs = retrieve_documents("berapa pesangon pekerja yang di PHK", k=5)
print_references(smoke_docs)

In [ ]:
# Phase 8 - Optional reranker.
# Default mati supaya notebook tetap ringan. Aktifkan kalau ingin download model reranker.
USE_RERANKER = False
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
reranker = None

if USE_RERANKER:
    from sentence_transformers import CrossEncoder
    reranker = CrossEncoder(RERANKER_MODEL_NAME, device=DEVICE)
    print("Reranker aktif:", RERANKER_MODEL_NAME)
else:
    print("Reranker neural nonaktif. Retrieval memakai RRF dense+BM25 + dedupe legal.")


def rerank_documents(query: str, docs: List[Dict[str, Any]], k: int = 6) -> List[Dict[str, Any]]:
    if reranker is None or not docs:
        return docs[:k]
    pairs = [(query, d["text"]) for d in docs]
    scores = reranker.predict(pairs)
    for doc, score in zip(docs, scores):
        doc["rerank_score"] = float(score)
    return sorted(docs, key=lambda x: x.get("rerank_score", 0.0), reverse=True)[:k]


In [ ]:
MAX_CONTEXT_DOCS = 4
MIN_QUERY_TERM_OVERLAP = 1
 
# FIX BUG 1:
# Expansion "pelanggaran berat" sebelumnya menyertakan kata "surat peringatan
# pertama kedua ketiga" sehingga retrieval malah menarik Pasal 52 ayat (1).
# Sekarang expansion dibagi jadi DUA rule terpisah berdasarkan jenis pelanggaran.
LEGAL_QUERY_EXPANSIONS = [
    {
        # Kondisi MENDESAK / berat: PHK seketika, TANPA SP 1,2,3
        # Target: Pasal 52 ayat (2) PP 35/2021
        "triggers": ["pelanggaran berat", "mendesak", "bersifat mendesak"],
        "expansion": (
            "pelanggaran bersifat mendesak uang pisah uang penggantian hak "
            "tidak mendapat pesangon tidak mendapat uang penghargaan masa kerja"
        ),
    },
    {
        # Kondisi BIASA: PHK karena pelanggaran ketentuan setelah SP 1, 2, 3
        # Target: Pasal 52 ayat (1) PP 35/2021
        # Rule ini HANYA aktif bila query sama sekali tidak menyebut "berat/mendesak"
        "triggers": ["surat peringatan", "sp pertama", "sp kedua", "sp ketiga"],
        "expansion": (
            "pelanggaran ketentuan surat peringatan pertama kedua ketiga "
            "pesangon nol koma lima uang penghargaan masa kerja uang penggantian hak"
        ),
    },
    {
        "triggers": ["pesangon", "phk", "pemutusan hubungan kerja"],
        "expansion": (
            "pemutusan hubungan kerja uang pesangon uang penghargaan masa kerja "
            "uang penggantian hak hak akibat pemutusan hubungan kerja"
        ),
    },
    {
        "triggers": ["pkwt", "kontrak"],
        "expansion": (
            "perjanjian kerja waktu tertentu kompensasi pkwt "
            "jangka waktu perpanjangan pembaruan"
        ),
    },
    {
        "triggers": ["alih daya", "outsourcing", "outsourced"],
        "expansion": (
            "alih daya perusahaan alih daya pekerja buruh "
            "hubungan kerja perlindungan upah kesejahteraan"
        ),
    },
    {
        "triggers": ["upah", "gaji", "tunjangan"],
        "expansion": (
            "upah gaji tunjangan tetap struktur skala upah "
            "upah minimum pembayaran upah"
        ),
    },
]
 
 
def assemble_context(docs, max_docs=MAX_CONTEXT_DOCS):
    assembled = []
    seen = set()
    for doc in docs:
        meta = doc.get("metadata", {})
        key = (
            meta.get("source_file", ""),
            meta.get("pasal_id", ""),
            meta.get("chunk_index", ""),
            doc.get("id", ""),
        )
        if key in seen:
            continue
        seen.add(key)
        assembled.append(doc)
        if len(assembled) >= max_docs:
            break
    return assembled
 
 
def query_terms(query):
    stopwords = {
        "yang", "dan", "atau", "karena", "dengan", "untuk", "pada", "dalam",
        "jika", "maka", "dari", "berapa", "apakah", "bagaimana", "dimana",
        "kapan", "siapa", "pekerja", "buruh", "pengusaha", "perusahaan", "hak",
        "nya", "itu", "ini", "ada", "dapat", "bisa", "oleh", "ke", "di", "atas",
    }
    terms = set()
    for token in re.findall(r"[a-zA-Z0-9]+", query.lower()):
        if len(token) <= 2 or token in stopwords:
            continue
        terms.add(token)
    return terms
 
 
def expand_query_terms(query):
    q = query.lower()
    expansions = []
    for rule in LEGAL_QUERY_EXPANSIONS:
        if any(trigger in q for trigger in rule["triggers"]):
            expansions.append(rule["expansion"])
    return " ".join([query] + expansions).strip()
 
 
def retrieval_queries(query):
    expanded = expand_query_terms(query)
    queries = [query]
    if expanded != query:
        queries.append(expanded)
    return queries
 
 
def doc_relevance_score(query, doc):
    expanded_query = expand_query_terms(query)
    terms = query_terms(expanded_query)
    if not terms:
        return 1.0
    meta = doc.get("metadata", {})
    haystack = " ".join([
        str(doc.get("text", "")),
        str(meta.get("citation_text", "")),
        str(meta.get("regulation_type", "")),
        str(meta.get("nomor", "")),
        str(meta.get("pasal_id", "")),
        str(meta.get("bab_title", "")),
        str(meta.get("bagian_title", "")),
    ]).lower()
    overlap = sum(1 for term in terms if term in haystack)
    score = overlap / max(len(terms), 1)
 
    original_terms = query_terms(query)
    specific_terms = {t for t in original_terms if t not in {"pesangon", "phk", "pemutusan", "hubungan", "kerja"}}
    specific_overlap = sum(1 for term in specific_terms if term in haystack)
    if specific_terms and specific_overlap == 0:
        score -= 0.25
 
    if "final_score" in doc:
        score += min(float(doc.get("final_score", 0.0)), 1.0) * 0.05
    elif "rrf_score" in doc:
        score += min(float(doc.get("rrf_score", 0.0)), 1.0) * 0.05
    return score
 
 
def has_original_specific_overlap(query, doc):
    original_terms = query_terms(query)
    specific_terms = {t for t in original_terms if t not in {"pesangon", "phk", "pemutusan", "hubungan", "kerja"}}
    if not specific_terms:
        return True
    meta = doc.get("metadata", {})
    haystack = " ".join([
        str(doc.get("text", "")),
        str(meta.get("citation_text", "")),
        str(meta.get("bab_title", "")),
        str(meta.get("bagian_title", "")),
    ]).lower()
    return any(term in haystack for term in specific_terms)
 
 
def filter_relevant_context(query, docs, max_docs=MAX_CONTEXT_DOCS):
    terms = query_terms(expand_query_terms(query))
    scored = []
    for doc in docs:
        if not has_original_specific_overlap(query, doc):
            continue
        score = doc_relevance_score(query, doc)
        overlap = int(round(max(score, 0) * max(len(terms), 1))) if terms else 1
        if (not terms or overlap >= MIN_QUERY_TERM_OVERLAP) and score > 0:
            doc["context_relevance_score"] = score
            scored.append(doc)
 
    if not scored:
        return docs[:1]
 
    scored = sorted(scored, key=lambda d: d.get("context_relevance_score", 0.0), reverse=True)
    return assemble_context(scored, max_docs=max_docs)
 
 
def extract_referenced_pasal_ids(text):
    refs = []
    for match in re.finditer(r"\bPasal\s+(\d+[A-Za-z]?)\b", text, flags=re.IGNORECASE):
        pasal = f"Pasal {match.group(1)}"
        if pasal not in refs:
            refs.append(pasal)
    return refs
 
 
def same_regulation(meta_a, meta_b):
    return (
        str(meta_a.get("regulation_type", "")).lower() == str(meta_b.get("regulation_type", "")).lower()
        and str(meta_a.get("nomor", "")).strip() == str(meta_b.get("nomor", "")).strip()
        and int(meta_a.get("publication_year") or meta_a.get("year") or 0)
        == int(meta_b.get("publication_year") or meta_b.get("year") or 0)
    )
 
 
def find_article_chunk_like(base_meta, pasal_id):
    for chunk in chunks:
        meta = normalize_metadata(chunk)
        if not same_regulation(base_meta, meta):
            continue
        if str(meta.get("pasal_id", "")).strip().lower() != pasal_id.lower():
            continue
        return {
            "id": chunk.get("id", ""),
            "text": chunk.get("display_text") or chunk.get("text") or "",
            "metadata": meta,
            "source": "referenced_article",
            "context_relevance_score": 0.99,
        }
    return None
 
 
# FIX BUG 2:
# Tambah parameter topic_guard dan topik_query untuk membatasi auto-expansion.
# Pasal yang disebut di dalam teks chunk hanya ditarik jika PASAL ITU SENDIRI
# muncul di hasil retrieval utama atau dipanggil langsung oleh satu ayat yang
# memang jadi fokus jawaban. Ini menghentikan Pasal 46/55 ikut masuk hanya
# karena disebut di batang teks Pasal 52.
def expand_context_with_referenced_articles(
    query, docs, max_docs=MAX_CONTEXT_DOCS, topic_guard=True
):
    expanded = assemble_context(docs, max_docs=max_docs)
    seen = {
        (d.get("metadata", {}).get("source_file", ""), d.get("metadata", {}).get("pasal_id", ""))
        for d in expanded
    }
 
    # Pasal yang sudah ada di hasil retrieval utama — ini "approved" untuk expand.
    primary_pasal_ids = {
        str(d.get("metadata", {}).get("pasal_id", "")).strip().lower()
        for d in expanded
        if d.get("source", "") != "referenced_article"
    }
 
    for doc in list(expanded):
        # Hanya expand referensi dari dokumen PRIMARY (bukan yang sudah di-inject).
        if doc.get("source") == "referenced_article":
            continue
 
        base_meta = doc.get("metadata", {})
        current_pasal = str(base_meta.get("pasal_id", "")).strip().lower()
 
        for pasal_id in extract_referenced_pasal_ids(doc.get("text", "")):
            if pasal_id.lower() == current_pasal:
                continue
 
            # FIX: Jika topic_guard aktif, hanya tarik pasal yang SUDAH ADA
            # di antara hasil retrieval utama. Pasal yang sekadar disebut
            # di teks (misal "sebagaimana Pasal 40 ayat 2") tidak otomatis ditarik
            # kecuali pasal itu memang relevan dengan query dan sudah di-retrieve.
            if topic_guard and pasal_id.lower() not in primary_pasal_ids:
                continue
 
            ref_doc = find_article_chunk_like(base_meta, pasal_id)
            if not ref_doc:
                continue
            key = (
                ref_doc.get("metadata", {}).get("source_file", ""),
                ref_doc.get("metadata", {}).get("pasal_id", ""),
            )
            if key in seen:
                continue
            expanded.append(ref_doc)
            seen.add(key)
            if len(expanded) >= max_docs:
                return expanded
    return expanded
 
 
def retrieve_context(query, k=MAX_CONTEXT_DOCS, fetch_k=50):
    max_docs = min(k, MAX_CONTEXT_DOCS)
    candidate_map = {}
    for q in retrieval_queries(query):
        hits = retrieve_documents(q, k=max(max_docs * 6, 18), fetch_k=fetch_k)
        for hit in hits:
            existing = candidate_map.get(hit["id"])
            if existing is None or hit.get("final_score", 0.0) > existing.get("final_score", 0.0):
                candidate_map[hit["id"]] = hit
 
    candidates = list(candidate_map.values())
    reranked = rerank_documents(expand_query_terms(query), candidates, k=max(max_docs * 6, 18))
    focused = filter_relevant_context(query, reranked, max_docs=max_docs)
    # topic_guard=True: hanya expand pasal yang sudah di-retrieve, bukan semua referensi silang
    return expand_context_with_referenced_articles(query, focused, max_docs=max_docs, topic_guard=True)


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

LLM_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"  # ganti ke model lokal/lebih besar kalau VRAM cukup
USE_4BIT = torch.cuda.is_available()

quantization_config = None
if USE_4BIT:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID, trust_remote_code=True)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    quantization_config=quantization_config,
    trust_remote_code=True,
)
print("LLM ready:", LLM_MODEL_ID)

In [ ]:
USE_LLM_KEYWORD_EXTRACTION = False  # rule-based default; LLM kecil sering bikin keyword typo.

TYPO_ARTIFACT_PATTERNS = [
    r"[\u0400-\u04FF\u0600-\u06FF]",
    r"pelanggar[a-z\u0400-\u04FF\u0600-\u06FF]*berat",
    r"pemrintah|tahu\s+20\d{2}|peringatann|berturut[- ]?tutur|mendesar|presangon|sebesr|pasial|ayta|pghganti|ketetuan|pekarja|pihargaan",
]

COMMON_ANSWER_FIXES = [
    (r"\bPemrintah\b", "Pemerintah"),
    (r"\bTahu\s+(20\d{2})\b", r"Tahun \1"),
    (r"\bperingatann\b", "peringatan"),
    (r"\bberturut[- ]?tutur\b", "berturut-turut"),
    (r"\bmendesar\b", "mendesak"),
    (r"\bpresangon\b", "pesangon"),
    (r"\bsebesr\b", "sebesar"),
    (r"\bPasial\b", "Pasal"),
    (r"\bayta\b", "ayat"),
    (r"\bPaial\b", "Pasal"),
    (r"\baya\b", "ayat"),
    (r"\bpghganti\b", "pengganti"),
    (r"\bketetuan\b", "ketentuan"),
    (r"\bpekarja\b", "pekerja"),
    (r"\bpihargaan\b", "penghargaan"),
]


def sanitize_output(text: str) -> str:
    emoji_pat = re.compile(r"[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF]+", flags=re.UNICODE)
    text = emoji_pat.sub("", text)
    text = re.sub(r"```.*?```", "", text, flags=re.DOTALL)
    text = re.sub(r"\*\*(.*?)\*\*", r"\1", text)
    text = re.sub(r"\*(.*?)\*", r"\1", text)
    text = re.sub(r"^#{1,6}\s*", "", text, flags=re.MULTILINE)
    text = re.sub(r"\s*\[(?:R\d+(?:\s*,\s*)?)+\]", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\b(?:Menurut|Berdasarkan)\s+(?:referensi|konteks|dokumen)\b[:,]?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"(?:\n\s*Referensi\s*:\s*)[\s\S]*$", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\n\s*[-=*]{3,}\s*\n", "\n", text)
    for pattern, repl in COMMON_ANSWER_FIXES:
        text = re.sub(pattern, repl, text, flags=re.IGNORECASE)
    return re.sub(r"\n{3,}", "\n\n", text).strip()


def answer_has_quality_issue(text: str) -> bool:
    if len(text.strip()) < 20:
        return True
    return any(re.search(pattern, text, flags=re.IGNORECASE) for pattern in TYPO_ARTIFACT_PATTERNS)


def generate_chat_text(messages: List[Dict[str, str]], max_new_tokens: int = 900, repair: bool = False) -> str:
    if hasattr(llm_tokenizer, "apply_chat_template"):
        try:
            prompt = llm_tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,
            )
        except TypeError:
            prompt = llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        prompt = "\n".join(f"{m['role'].upper()}: {m['content']}" for m in messages) + "\nASSISTANT:"

    inputs = llm_tokenizer(prompt, return_tensors="pt", padding=True, truncation=True, max_length=12000).to(llm_model.device)
    generation_kwargs = dict(
        **inputs,
        max_new_tokens=max_new_tokens,
        eos_token_id=llm_tokenizer.eos_token_id,
        pad_token_id=llm_tokenizer.eos_token_id,
    )
    if repair:
        generation_kwargs.update(do_sample=False)
    else:
        generation_kwargs.update(do_sample=True, temperature=0.04, top_p=0.9)

    outputs = llm_model.generate(**generation_kwargs)
    return llm_tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def extract_keywords(query: str) -> str:
    if not USE_LLM_KEYWORD_EXTRACTION:
        return query
    messages = [
        {"role": "system", "content": "Ekstrak kata kunci hukum penting dari pertanyaan user. Berikan HANYA kata kuncinya, pisahkan dengan koma, tanpa penjelasan."},
        {"role": "user", "content": query},
    ]
    keywords = sanitize_output(generate_chat_text(messages, max_new_tokens=48, repair=True))
    keywords = re.sub(r"^[Kk]ata\s+[Kk]unci\s*:\s*", "", keywords).strip()
    if answer_has_quality_issue(keywords):
        return query
    print("Kata kunci:", keywords)
    return keywords or query


def build_context(docs: List[Dict[str, Any]]) -> str:
    parts = []
    for i, doc in enumerate(docs, 1):
        meta = doc["metadata"]
        citation = compact_citation(meta)
        clean_text = str(doc["text"]).strip()
        parts.append(f"SUMBER HUKUM {i}: {citation}\n{clean_text}")
    return "\n\n".join(parts)


def _normalize_reg_type(value: str) -> str:
    value = str(value or "").lower()
    if "undang" in value or value == "uu":
        return "uu"
    if "pemerintah" in value or value == "pp":
        return "pp"
    if "presiden" in value or "perpres" in value:
        return "perpres"
    if "menteri" in value or "permen" in value:
        return "permen"
    return value


def validate_named_citations(answer: str, docs: List[Dict[str, Any]]) -> Dict[str, Any]:
    pattern = re.compile(
        r"(Undang-Undang|Peraturan Pemerintah|Peraturan Presiden|Peraturan Menteri(?:\s+Ketenagakerjaan)?|UU|PP|Perpres|Permen)"
        r"\s+(?:No\.?|Nomor)?\s*([A-Za-z0-9./-]+)\s+Tahun\s+(\d{4})",
        re.IGNORECASE,
    )
    cited: List[str] = []
    invalid: List[str] = []
 
    for reg_type, nomor, year in pattern.findall(answer):
        label = f"{reg_type} No. {nomor} Tahun {year}"
        cited.append(label)
        target_type = _normalize_reg_type(reg_type)
        found = False
        domain_ok = True
 
        for doc in docs:
            meta = doc.get("metadata", {})
            doc_type = _normalize_reg_type(meta.get("regulation_type", ""))
            doc_nomor = str(meta.get("nomor", "")).strip()
            doc_year = str(meta.get("publication_year") or meta.get("year") or "").strip()
 
            if doc_type == target_type and doc_nomor == str(nomor).strip() and doc_year == str(year):
                found = True
                # FIX: cek domain dokumen yang cocok.
                # Jika dokumen ada di context tapi bukan ketenagakerjaan → tandai invalid.
                if not is_ketenagakerjaan_doc(meta):
                    domain_ok = False
                break
 
        if not found or not domain_ok:
            invalid.append(label)
 
    out_of_scope = (
        "di luar lingkup dokumen hukum ketenagakerjaan" in answer.lower()
        or "tidak tersedia dalam database" in answer.lower()
    )
    quality_issue = answer_has_quality_issue(answer)
    ok = (not invalid) and (not quality_issue) and (bool(cited) or out_of_scope)
    return {
        "cited": sorted(set(cited)),
        "invalid": sorted(set(invalid)),
        "quality_issue": quality_issue,
        "ok": ok,
    }


def build_messages(query, docs, repair_note=""):
    context = build_context(docs)
 
    # FIX BUG 3: Tambah CONDITIONAL LOGIC GUARDRAIL di system prompt.
    # Ini instruksi eksplisit agar LLM tidak mencampuradukkan ayat (1) dan ayat (2)
    # dari Pasal 52 PP 35/2021, serta tidak mencampur kondisi lain yang serupa.
    conditional_logic_guardrail = """
ATURAN KRITIS — PEMISAHAN KONDISI HUKUM (WAJIB DIPATUHI):
Saat dokumen memuat BEBERAPA AYAT dengan kondisi berbeda dalam satu pasal yang sama,
kamu WAJIB mengidentifikasi ayat mana yang berlaku untuk kasus pengguna, lalu HANYA
gunakan konsekuensi hukum dari ayat tersebut. DILARANG KERAS menggabungkan kondisi
dari dua ayat berbeda menjadi satu narasi.
 
Contoh penerapan pada Pasal 52 PP No. 35 Tahun 2021:
  KONDISI A — Ayat (1): PHK karena pelanggaran ketentuan yang didahului SP ke-1, ke-2,
    dan ke-3. Konsekuensi: pesangon 0,5x, UPMK 1x, UPH. JANGAN sebut ini
    "pelanggaran berat". JANGAN sebut konsekuensi ini untuk kondisi mendesak.
  KONDISI B — Ayat (2)/(4): PHK karena pelanggaran bersifat mendesak (dahulu disebut
    pelanggaran berat). Dilakukan TANPA SP 1,2,3. Konsekuensi: TIDAK BERHAK PESANGON,
    TIDAK BERHAK UPMK, hanya berhak UPH dan Uang Pisah.
  LARANGAN ABSOLUT: Jangan pernah menulis "pelanggaran berat" lalu menyebut
    "pesangon 0,5x" — itu kontradiksi fatal. Pilih satu kondisi sesuai pertanyaan.
 
Prinsip ini berlaku umum untuk SEMUA pasal yang memiliki beberapa ayat dengan
kondisi berbeda di seluruh dokumen hukum yang tersedia.
"""
 
    system_prompt = (
        "Kamu adalah pakar hukum ketenagakerjaan Indonesia yang sangat teliti.\n"
        "Saat membaca dokumen hukum, kamu harus memahami bahwa setiap AYAT memiliki kondisi (syarat) dan konsekuensi yang BERBEDA.\n\n"
        "ATURAN UTAMA:\n"
        "1. JAWAB HANYA berdasarkan KONTEKS. DILARANG mengarang atau memakai pengetahuan luar.\n"
        "2. Langsung sebutkan sumber hukumnya secara natural, contoh: Peraturan Pemerintah No. 35 Tahun 2021, Pasal 52 ayat (2).\n"
        "   Sertakan nomor AYAT jika relevan — ini penting untuk membedakan kondisi hukum yang berbeda.\n"
        "3. DILARANG memakai kode [R1], [R2], SUMBER HUKUM 1, atau ID internal lain di jawaban.\n"
        "4. DILARANG menulis daftar referensi di dalam jawaban; sistem akan mencetak referensi terpisah.\n"
        "5. PENANGANAN KETERBATASAN INFORMASI:\n"
        "   a. Jika pertanyaan menggunakan istilah lama, bahasa awam, atau sinonim "
        "      (contoh: 'pelanggaran berat' = 'pelanggaran bersifat mendesak', "
        "      'kontrak' = 'PKWT', 'dipecat' = 'PHK'), JANGAN tolak pertanyaan. "
        "      Petakan ke istilah resmi dalam dokumen, sebutkan perubahan istilah "
        "      tersebut secara singkat di awal jawaban, lalu langsung jawab substansinya.\n"
        "   b. Hanya gunakan kalimat 'Maaf, informasi tersebut tidak tersedia dalam "
        "      database hukum ketenagakerjaan yang saya miliki.' jika setelah memetakan "
        "      semua kemungkinan sinonim pun tidak ada dokumen yang relevan sama sekali.\n"
        "   c. DILARANG KERAS memulai jawaban dengan kata 'Maaf' atau kalimat disclaimer "
        "      apapun jika konteks sudah tersedia. Langsung jawab substansinya.\n"
        "6. Selalu utamakan aturan terbaru atau aturan yang lebih spesifik jika ada perbedaan antar referensi.\n"
        "7. KETAT PADA KONTEKS: Abaikan dokumen atau pasal yang tidak relevan dengan substansi pertanyaan pengguna.\n"
        "8. SPESIFIK & AKURAT: Sebutkan peraturan, Pasal, beserta AYAT-nya dengan presisi. Jangan pernah mencampuradukkan konsekuensi antar ayat!\n"
        "9. KOMPREHENSIF: Sebutkan semua komponen hak, kewajiban, atau tata cara secara lengkap sesuai ayat yang diekstrak.\n"
        "   Jika berdasarkan pasal tersebut ada ketentuan 'tidak mendapat X', nyatakan pengecualian tersebut dengan tegas.\n"
        "10. Bahasa Indonesia harus rapi, baku, tanpa typo, tanpa campuran aksara asing, tanpa markdown, dan tidak berulang.\n"
        "11. Jika ragu, lebih baik jawab keterbatasan konteks daripada membuat pasal/tahun palsu.\n"
        f"{conditional_logic_guardrail}"
        f"{repair_note}"
    )
    user_prompt = f"KONTEKS REFERENSI HUKUM:\n{context}\n\nPERTANYAAN PENGGUNA:\n{query}"
    return [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]


def repair_answer(query: str, docs: List[Dict[str, Any]], bad_answer: str, max_new_tokens: int = 700) -> str:
    repair_note = (
        "\nMODE PERBAIKAN: Jawaban sebelumnya mengandung typo, aksara asing, atau sitasi tidak valid. "
        "Tulis ulang dari awal dengan bahasa Indonesia rapi. Jangan salin typo dari jawaban lama. "
        "Pastikan nomor dan tahun peraturan hanya diambil dari KONTEKS.\n"
    )
    messages = build_messages(query, docs, repair_note=repair_note)
    messages.append({"role": "user", "content": f"Jawaban lama yang harus diperbaiki, jangan disalin mentah:\n{bad_answer}"})
    return sanitize_output(generate_chat_text(messages, max_new_tokens=max_new_tokens, repair=True))


def _query_terms(query: str) -> set[str]:
    stopwords = {
        "yang", "dan", "atau", "karena", "dengan", "untuk", "pada", "dalam", "jika", "maka",
        "berapa", "apakah", "pekerja", "buruh", "di", "ke", "dari", "atas", "hak", "nya", "ia",
    }
    return {t for t in re.findall(r"[a-zA-Z0-9]+", query.lower()) if len(t) > 2 and t not in stopwords}


def _split_sentences(text: str) -> List[str]:
    text = re.sub(r"\s+", " ", text).strip()
    parts = re.split(r"(?<=[.!?])\s+|(?=\([0-9]+\)\s)|(?=\b[a-z]\.)", text)
    return [p.strip() for p in parts if len(p.strip()) > 25]


def extractive_fallback_answer(query: str, docs: List[Dict[str, Any]], max_sentences: int = 8) -> str:
    terms = _query_terms(query)
    if not docs:
        return "Maaf, informasi tersebut tidak tersedia dalam database hukum ketenagakerjaan yang saya miliki."

    selected = []
    seen = set()
    for doc in docs:
        citation = compact_citation(doc.get("metadata", {}))
        sentences = _split_sentences(doc.get("text", ""))
        scored = []
        for sentence in sentences:
            low = sentence.lower()
            score = sum(1 for term in terms if term in low)
            if "pasal" in low:
                score += 1
            if score > 0:
                scored.append((score, sentence))
        scored.sort(key=lambda x: x[0], reverse=True)
        for _, sentence in scored[:3]:
            key = (citation, sentence[:120])
            if key in seen:
                continue
            seen.add(key)
            selected.append((citation, sentence))
            if len(selected) >= max_sentences:
                break
        if len(selected) >= max_sentences:
            break

    if not selected:
        top = docs[0]
        citation = compact_citation(top.get("metadata", {}))
        excerpt = re.sub(r"\s+", " ", top.get("text", "")).strip()[:900]
        return f"{citation} memuat ketentuan berikut: {excerpt}"

    grouped = []
    used_citations = set()
    for citation, sentence in selected:
        prefix = citation if citation not in used_citations else "Ketentuan tersebut"
        used_citations.add(citation)
        grouped.append(f"{prefix} menyatakan bahwa {sentence}")

    return "\n\n".join(grouped)


def generate_answer(query: str, k: int = 6, max_new_tokens: int = 800) -> Dict[str, Any]:
    optimized_query = extract_keywords(query)
    docs = retrieve_context(optimized_query, k=k)
    messages = build_messages(query, docs)
    answer = sanitize_output(generate_chat_text(messages, max_new_tokens=max_new_tokens))
    validation = validate_named_citations(answer, docs)
 
    if not validation["ok"]:
        repaired = repair_answer(query, docs, answer, max_new_tokens=max_new_tokens)
        repaired_validation = validate_named_citations(repaired, docs)
        if repaired_validation["ok"]:
            answer = repaired
            validation = repaired_validation
        else:
            answer = extractive_fallback_answer(query, docs)
            validation = validate_named_citations(answer, docs)
            validation["fallback"] = "extractive"
 
    # FIX BUG D: pisahkan "semua docs yang diambil" vs "docs yang benar-benar dikutip".
    cited = cited_docs_only(docs, validation)
 
    return {
        "query": query,
        "optimized_query": optimized_query,
        "answer": answer,
        "references": docs,         # semua retrieved docs (untuk debugging)
        "cited_references": cited,  # HANYA docs yang dikutip LLM (untuk ditampilkan ke user)
        "validation": validation,
    }
 


result = generate_answer("Jika pekerja di-PHK karena pelanggaran berat, berapa pesangonnya?", k=3)
print(result["answer"])
print("\nReferensi:")
print_references(result["references"])
print("\nValidation:", result["validation"])


In [ ]:
import pandas as pd
eval_questions = [
    "Jika pekerja di-PHK karena pelanggaran berat, berapa pesangonnya?",
    "Apa kewajiban pengusaha terkait alat pelindung diri?",
    "Apa prinsip pelaksanaan SOP administrasi pemerintahan?",
    "Bagaimana aturan keselamatan dan kesehatan kerja lingkungan kerja?",
]

rows = []
for q in eval_questions:
    out = generate_answer(q, k=6, max_new_tokens=700)
    rows.append({
        "question": q,
        "answer": out["answer"],
        "contexts": [d["text"] for d in out["references"]],
        "reference_citations": [build_reference(d["metadata"], i) for i, d in enumerate(out["references"], 1)],
        "citations_used": out["validation"]["cited"],
        "invalid_citations": out["validation"]["invalid"],
    })

# Format print tanpa emoji untuk membaca jawaban secara penuh
print("\n" + "#"*80 + "\n   HASIL EVALUASI RAG KESELURUHAN\n" + "#"*80 + "\n")

for i, row in enumerate(rows, 1):
    print("="*80)
    print(f"[{i}] PERTANYAAN:")
    print(f"   {row['question']}")
    print("\nJAWABAN MODEL (FULL TEXT):")
    print(row['answer'].strip())
    print(f"\nSITASI DIPAKAI : {row['citations_used']}")
    print(f"SITASI INVALID : {row['invalid_citations']}")
    print("DOKUMEN YANG BERHASIL DI-RETRIEVE (k=6):")
    for ref in row['reference_citations']:
        print(f"  - {ref}")
    print("="*80 + "\n")

# Dataframe tetap disimpan di memori untuk kebutuhan metrik RAGAS
eval_df = pd.DataFrame(rows)


In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import answer_relevancy, faithfulness, context_precision, context_recall

# Isi ground_truth kalau mau context_recall bermakna. Kalau kosong, pakai metrik tanpa context_recall.
ragas_rows = []
for row in rows:
    ragas_rows.append({
        "question": row["question"],
        "answer": row["answer"],
        "contexts": row["contexts"],
        "ground_truth": "",
    })

dataset = Dataset.from_list(ragas_rows)
metrics = [faithfulness, answer_relevancy, context_precision]
# Tambahkan context_recall hanya jika ground_truth diisi manual.
# metrics.append(context_recall)

# Butuh konfigurasi LLM/embedding RAGAS sesuai environment. Jalankan setelah env RAGAS siap.
# score = evaluate(dataset, metrics=metrics)
# score.to_pandas()

## Perbandingan dengan backup `03_Phase_10_11_Generation_Eval.ipynb`

Yang dipertahankan dari notebook lama:
- grounding ketat ke konteks;
- hybrid retrieval dense + BM25;
- citation validation;
- opsi model 4-bit untuk GPU.

Yang diubah:
- sumber data lama diganti ke `../data/processed_chunks_ringan_pasal_chroma_ready.json`;
- teks embedding memakai `embedding_text`, bukan `text` mentah;
- dokumen yang masuk prompt memakai `display_text`;
- referensi memakai `citation_text` ringkas dan jawaban menyebut sumber hukum secara natural, sehingga lebih gampang dibaca;
- filter lama `section_type == batang_tubuh` dihapus karena schema chunk baru tidak punya field itu. Kualitas dikontrol saat finalisasi melalui quarantine dan split pasal.
